# Feature Engineering and Preprocessing: Ames Housing Pipeline

## Introduction
This notebook focuses on one of the most practical parts of applied machine learning: turning a messy tabular dataset into a clean modeling pipeline. The emphasis is on handling numeric and categorical variables correctly and preventing data leakage.

## Project Goal
Build a reproducible sklearn pipeline for a mixed-type dataset, compare a baseline feature set against a refined subset, and explain why preprocessing design matters.

## Machine Learning Concepts Used
- Feature Engineering
- Missing Value Imputation
- Categorical Encoding
- Scaling
- ColumnTransformer
- Pipeline
- Data Leakage Prevention
- Regression

## Dataset
`sklearn.datasets.fetch_openml(name='house_prices', as_frame=True)`

## Step 1: Import libraries

**What this section is doing**  
Import the tools required for real-world tabular preprocessing. This includes imputers, encoders, scalers, and sklearn pipeline components.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [1]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 200)

## Step 2: Load the dataset

**What this section is doing**  
Load the Ames housing dataset from OpenML. This dataset is more realistic than toy data because it contains missing values, mixed types, and many columns.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [2]:
housing = fetch_openml(name="house_prices", as_frame=True)
df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,None,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,'Wd Sdng','Wd Shng',None,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Step 3: Inspect missingness and data types

**What this section is doing**  
Before preprocessing, identify which variables are incomplete and how many numeric vs categorical columns are present.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [3]:
missing_df = df.isna().sum().sort_values(ascending=False).to_frame("missing_count")
display(missing_df.head(20))

dtype_summary = df.dtypes.astype(str).value_counts().to_frame("count")
display(dtype_summary)

,missing_count
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
FireplaceQu,690
LotFrontage,259
GarageFinish,81
GarageQual,81
GarageYrBlt,81
GarageType,81


,count
object,43
int64,35
float64,3


## Step 4: Select a manageable feature subset

**What this section is doing**  
For a fundamentals notebook, it is better to use a focused feature set and explain it well than to include dozens of columns without interpretation.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [4]:
selected_features = [
    "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "GrLivArea", "BedroomAbvGr", "FullBath", "GarageCars",
    "Neighborhood", "HouseStyle", "ExterQual"
]
target_col = "SalePrice"

data = df[selected_features + [target_col]].copy()
X = data[selected_features]
y = data[target_col].astype(float)

display(X.head())

,LotArea,OverallQual,OverallCond,YearBuilt,GrLivArea,BedroomAbvGr,FullBath,GarageCars,Neighborhood,HouseStyle,ExterQual
0,8450,7,5,2003,1710,3,2,2,CollgCr,2Story,Gd
1,9600,6,8,1976,1262,3,2,2,Veenker,1Story,TA
2,11250,7,5,2001,1786,3,2,2,CollgCr,2Story,Gd
3,9550,7,5,1915,1717,3,1,3,Crawfor,2Story,TA
4,14260,8,5,2000,2198,4,2,3,NoRidge,2Story,Gd


## Step 5: Separate numeric and categorical variables

**What this section is doing**  
A mixed-type dataset needs different preprocessing rules for different variable types.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [5]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:")
display(pd.Series(numeric_features))

print("Categorical features:")
display(pd.Series(categorical_features))

print("Missing values in selected set:")
display(X.isna().sum())

Numeric features:


,0
0,LotArea
1,OverallQual
2,OverallCond
3,YearBuilt
4,GrLivArea
5,BedroomAbvGr
6,FullBath
7,GarageCars


Categorical features:


,0
0,Neighborhood
1,HouseStyle
2,ExterQual


Missing values in selected set:


,0
LotArea,0
OverallQual,0
OverallCond,0
YearBuilt,0
GrLivArea,0
BedroomAbvGr,0
FullBath,0
GarageCars,0
Neighborhood,0
HouseStyle,0


## Step 6: Check numeric feature relevance

**What this section is doing**  
Before building the pipeline, inspect a rough relevance signal for the numeric variables using correlation with the target.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [6]:
corr_df = data[numeric_features + [target_col]].corr(numeric_only=True)[target_col].sort_values(ascending=False)
display(corr_df.to_frame("correlation_with_target"))

,correlation_with_target
SalePrice,1.000000
OverallQual,0.790982
GrLivArea,0.708624
GarageCars,0.640409
FullBath,0.560664
YearBuilt,0.522897
LotArea,0.263843
BedroomAbvGr,0.168213
OverallCond,-0.077856


## Step 7: Create train and test sets

**What this section is doing**  
Split the data before any preprocessing is fit. This is one of the most important anti-leakage rules in machine learning.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (1168, 11)
X_test shape : (292, 11)


## Step 8: Build preprocessing pipelines

**What this section is doing**  
Use separate pipelines for numeric and categorical variables so each type is handled appropriately.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [8]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['LotArea', 'OverallQual', 'OverallCond',
                                  'YearBuilt', 'GrLivArea', 'BedroomAbvGr',
                                  'FullBath', 'GarageCars']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Neighborhood', 'HouseStyle', 'ExterQual'])])

## Step 9: Create the full model pipeline

**What this section is doing**  
Combine preprocessing and model fitting into one sklearn Pipeline so the workflow is reproducible and less error-prone.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [9]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)
print("Pipeline training complete.")

Pipeline training complete.


## Step 10: Evaluate the baseline pipeline

**What this section is doing**  
Generate predictions and assess regression performance using multiple metrics.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [10]:
y_pred = model.predict(X_test)

metrics_df = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R^2"],
    "value": [
        mean_absolute_error(y_test, y_pred),
        mean_squared_error(y_test, y_pred) ** 0.5,
        r2_score(y_test, y_pred)
    ]
})

display(metrics_df)

,metric,value
0,MAE,20056.842609
1,RMSE,33532.579208
2,R^2,0.853405


## Step 11: Refine the feature set and retrain

**What this section is doing**  
Test whether a slightly smaller, more intentionally chosen feature set changes performance. For mixed-type data, this can make the workflow easier to explain without losing too much predictive value.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [11]:
selected_numeric = (
    corr_df.drop(target_col, errors="ignore")
    .abs()
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)
selected_categorical = ["Neighborhood", "HouseStyle", "ExterQual"]

selected_features_refined = selected_numeric + selected_categorical
print("Refined feature set:", selected_features_refined)

X_refined = data[selected_features_refined]
num_refined = X_refined.select_dtypes(include=["number"]).columns.tolist()
cat_refined = X_refined.select_dtypes(exclude=["number"]).columns.tolist()

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_refined, y, test_size=0.2, random_state=42
)

refined_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_refined),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_refined)
])

refined_model = Pipeline(steps=[
    ("preprocessor", refined_preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

refined_model.fit(Xr_train, yr_train)
yr_pred = refined_model.predict(Xr_test)

comparison_df = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R^2"],
    "baseline_model": [
        mean_absolute_error(y_test, y_pred),
        mean_squared_error(y_test, y_pred) ** 0.5,
        r2_score(y_test, y_pred)
    ],
    "refined_feature_model": [
        mean_absolute_error(yr_test, yr_pred),
        mean_squared_error(yr_test, yr_pred) ** 0.5,
        r2_score(yr_test, yr_pred)
    ]
})

display(comparison_df)

Refined feature set: ['OverallQual', 'GrLivArea', 'GarageCars', 'FullBath', 'Neighborhood', 'HouseStyle', 'ExterQual']


,metric,baseline_model,refined_feature_model
0,MAE,20056.842609,21505.365529
1,RMSE,33532.579208,34885.517991
2,R^2,0.853405,0.841337


## Step 12: Final analysis and next steps

**What this section is doing**  
Finish by explaining why the pipeline is professional, what the refined feature experiment showed, and what stronger future versions might include.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

### Performance Comparison Summary

From the `comparison_df`, we can observe the following metrics for both the baseline and refined feature models:

*   **Mean Absolute Error (MAE)**:
    *   Baseline Model: `20056.84`
    *   Refined Feature Model: `21505.37`
*   **Root Mean Squared Error (RMSE)**:
    *   Baseline Model: `33532.58`
    *   Refined Feature Model: `34885.52`
*   **R-squared (R^2)**:
    *   Baseline Model: `0.853`
    *   Refined Feature Model: `0.841`

The baseline model, which used a larger set of selected features, performed slightly better than the refined feature model across all three metrics (MAE, RMSE, and R^2). The MAE and RMSE are lower for the baseline model, indicating smaller prediction errors, and its R^2 value is higher, suggesting it explains a larger proportion of the variance in the target variable. This outcome is not unexpected, as the refined model explicitly removed features with lower absolute correlation to the target variable (`LotArea`, `OverallCond`, `YearBuilt`, `BedroomAbvGr`), which, while individually less correlated, might have contributed valuable information when combined, or their absence led to a slight degradation in performance.

### Next Steps for Model Improvement and Exploration

To further enhance the model's predictive power and robustness, consider the following:

1.  **Try Different Regression Algorithms**: Explore more complex or different types of models such as `RandomForestRegressor`, `GradientBoostingRegressor`, `XGBoost`, or `LightGBM`. These models often capture non-linear relationships and interactions between features more effectively.
2.  **Advanced Feature Engineering**: Delve deeper into feature engineering. This could include creating interaction terms (e.g., `OverallQual` * `GrLivArea`), polynomial features, or extracting more information from existing categorical features (e.g., target encoding for `Neighborhood`).
3.  **Hyperparameter Tuning**: For the chosen models (including the current Ridge regression), conduct systematic hyperparameter tuning using techniques like `GridSearchCV` or `RandomizedSearchCV` to find the optimal combination of parameters that maximize performance.
4.  **Cross-Validation**: Implement K-fold cross-validation during training to obtain a more robust estimate of the model's performance and reduce the risk of overfitting to a single train-test split.
5.  **Experiment with Imputation/Encoding Strategies**: Investigate alternative strategies for handling missing values (e.g., more sophisticated imputers like KNNImputer) and categorical features (e.g., `TargetEncoder` or `OrdinalEncoder` if feature ordinality can be established).
6.  **Error Analysis**: Conduct a thorough error analysis on the predictions to identify patterns in where the model performs poorly. This can reveal specific data points or feature ranges that might require special attention or further feature engineering.